<img src=https://aggis.org/images/aggis.ico> 

# Airbnb Exploration Tools
## `airbnb.ipynb`
### aggis.org/files/notebooks/.
Adapted from assignment submission for UCLA GEOG 412: Programming for Geospatial Data Science, Unit 4.
<hr/>

#### 📦 Packages

In [ ]:
!pip install shapely==1.8.5.post1 pandas geopandas geoplot mapclassify contextily
!sudo apt install -y libspatialindex-dev
!pip install rtree

import pandas as pd
import geopandas as gpd
import math
import numpy as np
import contextily as cx
import matplotlib.pyplot as plt
import matplotlib.colors as cl
import mapclassify
import ipywidgets as widgets
from IPython.display import display, clear_output
import fiona
import rtree
import geoplot as gplt
from geopy.geocoders import Nominatim
from shapely.geometry import shape, Point

#### 📊 Data

In [ ]:
# For Google Colab ========================================================|
!wget = https://aggis.org/files/notebooks/data/listings.csv               #|
!wget = https://aggis.org/files/notebooks/data/neighbourhoods.geojson     #|
!wget = https://aggis.org/files/notebooks/data/ams_listings.csv           #|
!wget = https://aggis.org/files/notebooks/data/ams_neighbourhoods.geojson #|
listings_path = 'listings.csv'                                            #|
neighbourhoods_path = 'neighbourhoods.geojson'                            #|
ams_listings_path = 'ams_listings.csv'                                    #|
ams_neighbourhoods_path = 'ams_neighbourhoods.geojson'                    #|
# =========================================================================|
# For Jupyter Notebook ====================================================|
# (local clone of the repo)                                                |
#listings_path = 'data/listings.csv'                                      #|
#neighbourhoods_path = 'data/neighbourhoods.geojson'                      #|  
#ams_listings_path = 'data/ams_listings.csv'                              #|
#ams_neighbourhoods_path = 'data/ams_neighbourhoods.geojson'              #|
# =========================================================================|

listings = pd.read_csv(listings_path)
neighbourhoods = gpd.read_file(neighbourhoods_path)
ams_listings = pd.read_csv(ams_listings_path)
ams_neighbourhoods = gpd.read_file(ams_neighbourhoods_path)

#### 🗺️ Map: Average Nightly Rate of Airbnb Rental, Los Angeles County Communities

In [ ]:
listings = gpd.GeoDataFrame(
    listings,
    geometry=gpd.points_from_xy(listings.longitude, listings.latitude),
    crs=4326) # pd df to gpd gdf

listings = listings.drop(columns=['neighbourhood','neighbourhood_group'])

nhoods_joined = neighbourhoods.sjoin(listings,how='left')

nhoods_joined["count"] = 1

nhoods_joined = nhoods_joined.groupby('neighbourhood').agg({'geometry':'first','price':'mean'})
nhoods_joined = nhoods_joined.set_geometry("geometry")
nhoods_joined = nhoods_joined.set_crs(epsg=4326)

fig, ax = plt.subplots(figsize=(12,12))
ax.set_title('LA Communities by Avg AirBnB Rate',fontsize=24)
ax.set_axis_off()

nhoods_map = nhoods_joined.plot(
    ax=ax,
    column="price",
    legend=True,
    figsize=(12,12),
    cmap="Reds",
    scheme="NaturalBreaks",
    k=5,
    edgecolor='black',
    linewidth=0.5)
cx.add_basemap(nhoods_map,crs=nhoods_joined.crs.to_string(),source=cx.providers.Esri.WorldStreetMap)

nhoods_joined.sort_values(by="price",ascending=False)

#### 🔨 Tool 1: Airbnb in Amsterdam, Region-Wide Data Exploration Tool

In [ ]:
ams_listings = gpd.GeoDataFrame(ams_listings,geometry=gpd.points_from_xy(ams_listings.longitude, ams_listings.latitude),crs=4326) # pd df to gpd gdf

# vis 1

ams_listings_reproj = ams_listings.to_crs(crs="28992")

fig, ax = plt.subplots(figsize=(12,12))
ax.set_title("AirBnB's in Amsterdam",fontsize=24)
ax.set_axis_off()
ams_listings_map = ams_listings_reproj.plot(
  ax=ax,
  figsize=(12,12),
  markersize=0.1)
cx.add_basemap(ams_listings_map,crs=ams_listings_reproj.crs,source=cx.providers.Esri.WorldStreetMap)

# vis 2

ams_listings = ams_listings.drop(columns=['neighbourhood','neighbourhood_group'])

ams_nhoods_joined = ams_neighbourhoods.sjoin(ams_listings,how='left')
ams_nhoods_joined["count"] = 1
ams_nhoods_joined = ams_nhoods_joined.groupby('neighbourhood').agg({'geometry':'first','price':'mean'})

ams_nhoods_joined = ams_nhoods_joined.set_geometry("geometry")
ams_nhoods_joined = ams_nhoods_joined.set_crs(epsg=4326)
ams_nhoods_reproj = ams_nhoods_joined.to_crs(crs="28992")

fig, ax = plt.subplots(figsize=(12,12))
ax.set_title('Amsterdam Neighbourhoods by Avg AirBnB Rate',fontsize=24)
ax.set_axis_off()
ams_nhoods_map = ams_nhoods_reproj.plot(
    ax=ax,
    column="price",
    legend=True,
    figsize=(12,12),
    cmap="Reds",
    scheme="NaturalBreaks",
    k=5,
    edgecolor='black',
    linewidth=0.5)
cx.add_basemap(ams_nhoods_map,crs=ams_nhoods_reproj.crs.to_string(),source=cx.providers.Esri.WorldStreetMap)

# vis 3

ams_listings_kd = gplt.kdeplot(ams_listings,figsize=(12,12),shade=True,clip=ams_neighbourhoods.dissolve())
gplt.polyplot(ams_neighbourhoods, zorder=1, ax=ams_listings_kd)

# vis 4

ams_nhoods_joined.hist(
      column='price',
      figsize=(12,6))

#### 🔨 Tool 2: Airbnb in Amsterdam,  Local Area Analytics Tool

**Input geocoded places must be in Amsterdam, The Netherlands**

*some example searches*
*   Bunk Hotel Amsterdam
*   Van Gogh Museum Amsterdam
*   Amsterdam Centraal Station
*   Anne Frank House Amsterdam
*   Vondelpark Amsterdam

**FOR THIS SCRIPT TO WORK, MAKE SURE THERE IS NO EXISTING 'listing.csv' FILE IN MEMORY.**

In [ ]:
ams_listings = gpd.GeoDataFrame(ams_listings,geometry=gpd.points_from_xy(ams_listings.longitude, ams_listings.latitude),crs=4326) # pd df to gpd gdf
ams_listings = ams_listings.set_geometry("geometry",crs="EPSG:4326") # set geo
ams_listings = ams_listings.set_crs(epsg=4326) # set crs
ams_listings.crs = "EPSG:4326" # direct crs
ams_listings = gpd.GeoDataFrame(ams_listings,geometry=ams_listings.geometry,crs=4326) # confirm

# geocoder
geocoder = Nominatim(user_agent='local_area_analysis')

# dropdown selectors
address_field = widgets.Text(value=input("Address: "),placeholder="Address: ",description="Address: ",disabled=False)
buffer_field = widgets.FloatText(value=float(input("Radius in meters (500m - 10,000m): ")),description="Radius <m>: ",disabled=False)
go_button = widgets.ToggleButton(value=False,description="Go!",disabled=False,button_style="",tooltip="Description",icon="")

# grid
grid = widgets.GridspecLayout(1,3,height="60px")
grid[0,0] = address_field
grid[0,1] = buffer_field
grid[0,2] = go_button

# on change event / function
def on_change(event=None):
  clear_output()

  if buffer_field.value < 500:
    print("Please enter a radius of more than 500 meters.")
    display(grid)
  elif buffer_field.value > 10000:
    print("Please enter a radius of less than 10 kilometers.")
    display(grid)
  else:
    print("Selected place: ",address_field.value)
    print("Defined radius: ",buffer_field.value)

    user_place = geocoder.geocode(address_field.value)
    user_pt = {"address":[user_place.address], "geometry":[Point(user_place.longitude, user_place.latitude)]}
    user_gdf = gpd.GeoDataFrame(user_pt, crs=4326)

    user_gdf_3310 = user_gdf.to_crs(3310) # reproject for buffering
    buffer = user_gdf_3310.buffer(buffer_field.value)
    buffer_df = gpd.GeoDataFrame(buffer,geometry=buffer)
    buffer_df_dissolved = buffer_df.dissolve()

    buffer_4326 = buffer_df_dissolved.to_crs(crs="4326") # back to 4326 for sjoin
    buffer_4326 = buffer_4326.set_geometry("geometry",crs="EPSG:4326") # set geo
    buffer_4326 = buffer_4326.set_crs(epsg=4326) # set crs
    buffer_4326.crs = "EPSG:4326" # direct crs
    buffer_4326 = gpd.GeoDataFrame(buffer_4326, geometry=buffer_4326.geometry, crs=4326) # confirm

    buffer_4326_join = gpd.sjoin(left_df=ams_listings,right_df=buffer_4326,how="inner",predicate="intersects")
    buffer_4326_join = buffer_4326_join.set_geometry("geometry",crs="EPSG:4326") # set geo
    buffer_4326_join = buffer_4326_join.set_crs(epsg=4326) # set crs
    buffer_4326_join.crs = "EPSG:4326" # direct crs
    buffer_4326_join = gpd.GeoDataFrame(buffer_4326_join, geometry=buffer_4326_join.geometry, crs=4326) # confirm

    buffer_to_map = buffer_4326_join.to_crs(crs="28992") # project to local dutch
    buffer_to_map = buffer_to_map.set_geometry("geometry",crs="EPSG:28992") # set geo
    buffer_to_map = buffer_to_map.set_crs(epsg=28992) # set crs
    buffer_to_map.crs = "EPSG:28992" # direct crs
    buffer_to_map = gpd.GeoDataFrame(buffer_to_map, geometry=buffer_to_map.geometry, crs=28992) # confirm

    fig, ax = plt.subplots(figsize=(16,6))
    ax.set_title("AirBnbs within " + str(buffer_field.value) + " meters of " + str(address_field.value),fontsize=16)
    ax.set_axis_off()
    local_map = buffer_to_map.plot(ax=ax,color="Red",figsize=(16,16),markersize=0.3) # plot
    cx.add_basemap(local_map,crs=buffer_to_map.crs,source=cx.providers.Esri.WorldStreetMap)

    print("There are " + str(len(buffer_to_map.index)) + " AirBnB rentals active within " + str(buffer_field.value) + " meters of " + str(address_field.value) + ".") # count

    local_avg_price = buffer_to_map["price"].mean(axis=0) # price
    local_avg_price = math.trunc(local_avg_price)
    region_avg_price = ams_listings["price"].mean(axis=0)
    region_avg_price = math.trunc(region_avg_price)
    print("These " + str(len(buffer_to_map.index)) + " rentals cost " + str(local_avg_price) + " USD per night, on average.")
    print("Compare this to the average nightly rate for all of Amsterdam: " + str(region_avg_price) + " USD per night.")

    local_rpm = buffer_to_map["reviews_per_month"].sum(axis=0) # activity
    local_rpm = math.trunc(local_rpm)
    region_rpm = ams_listings["reviews_per_month"].sum(axis=0)
    region_rpm = math.trunc(region_rpm)
    pct_city_activity = ( local_rpm / region_rpm ) * 100
    pct_city_activity = math.trunc(pct_city_activity)
    print("Together these rentals receive up to " + str(local_rpm) + " reviews per month, on average.")
    print("That amounts to " + str(pct_city_activity) + "% of the city's monthly user-review activity on AirBnB.")
    print("Edit your search parameters and click Go! to view a new region.")

    display(grid)

# observe
go_button.observe(on_change,names="value")

# start
display(grid)